# Phase A：StudentPolicy（对称 BC 先验 + 碰撞建图 + 重规划）

In [ ]:
# 单元格 1：准备环境（克隆仓库、切换目录）
import os, sys
ROOT = '/content/robotics_learning_group_selection'
if not os.path.exists(ROOT):
    !git clone -q https://github.com/zhangyunzhen2027/robotics_learning_group_selection.git {ROOT}
%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
!pip -q install numpy matplotlib
print('ready:', sorted(os.listdir(ROOT)))

**单元格 2：写入策略代码 src/policy.py（StudentPolicy 在这里）**

In [ ]:
%%writefile src/policy.py
import heapq

import numpy as np


def _unit(a):
    a = np.asarray(a, dtype=float)
    n = np.linalg.norm(a)
    if n < 1e-8:
        return np.zeros(2, dtype=np.float32)
    return (a / n).astype(np.float32)


class LinearPolicy:
    """Provided baseline: least-squares linear behavioral cloning."""
    def __init__(self, X, Y):
        Xb = np.concatenate([X, np.ones((len(X), 1))], axis=1)
        self.W = np.linalg.pinv(Xb) @ Y

    def act(self, obs):
        x = np.concatenate([obs, [1.0]])
        a = x @ self.W
        n = np.linalg.norm(a)
        if n > 1.0:
            a = a / n
        elif n > 1e-8:
            a = a / n
        return a.astype(np.float32)


# --------------------------------------------------------------------------- #
# Symmetry-aware behavioural cloning
# --------------------------------------------------------------------------- #
# The 8 symmetries of the square (dihedral group D4) as 2x2 matrices.
D4 = [np.array(m, dtype=float) for m in (
    [[1, 0], [0, 1]], [[0, -1], [1, 0]], [[-1, 0], [0, -1]], [[0, 1], [-1, 0]],   # rotations
    [[1, 0], [0, -1]], [[-1, 0], [0, 1]], [[0, 1], [1, 0]], [[0, -1], [-1, 0]],   # reflections
)]


def d4_augment(X, Y):
    """Start/target are uniform in a square and blocks are axis-aligned with iid
    width/height, so the scenario distribution is invariant under D4. A demo
    (obs, action) therefore stays valid after applying the same g in D4 to both:
    (g obs, g action). This multiplies the demo budget by 8 for free."""
    X, Y = np.asarray(X, float), np.asarray(Y, float)
    return (np.concatenate([X @ g.T for g in D4]).astype(np.float32),
            np.concatenate([Y @ g.T for g in D4]).astype(np.float32))


class SymmetricLinearPolicy:
    """Linear BC fitted on D4-augmented demos, without a bias term.

    A bias is forbidden by the symmetry (E[a | -obs] = -E[a | obs]); estimating it
    only adds variance, and near the goal (obs -> 0) a non-zero bias dominates the
    action and makes the robot circle around the goal.
    """

    def __init__(self, X, Y):
        Xa, Ya = d4_augment(X, Y)
        self.W = np.linalg.pinv(Xa) @ Ya

    def act(self, obs):
        return _unit(np.asarray(obs, dtype=float) @ self.W)


# --------------------------------------------------------------------------- #
# Turning the observation stream into events
# --------------------------------------------------------------------------- #
class ContactTracker:
    """Extracts the hidden signals in a stream of relative observations.

    obs = target - robot. Within an episode the target is fixed, so
      * robot position in a goal-centred frame is  p = -obs  (no drift: every
        observation is a fresh measurement), and
      * a normal step changes obs by ~step_size; a blocked step changes it by ~0
        -> contact / collision sensing for free.
    The evaluator never calls reset(), so a new episode is detected from
      * the previous observation being inside the success radius, or
      * obs jumping by much more than one step.
    """

    def __init__(self, step=0.08, success_dist=0.05, jump=2.0, stall=0.5):
        self.step = step
        self.success_dist = success_dist
        self.jump = jump * step
        self.stall = stall * step
        self.prev_obs = None
        self.prev_act = None

    def observe(self, obs):
        obs = np.asarray(obs, dtype=float)
        ev = {"new_episode": False, "collided": False, "hit_point": None}
        if self.prev_obs is None or self.prev_act is None:
            ev["new_episode"] = True
        else:
            moved = float(np.linalg.norm(obs - self.prev_obs))
            if np.linalg.norm(self.prev_obs) < self.success_dist or moved > self.jump:
                ev["new_episode"] = True
            elif moved < self.stall and np.linalg.norm(self.prev_act) > 1e-6:
                ev["collided"] = True
                # the point we tried to reach (goal frame) is inside an obstacle
                ev["hit_point"] = -self.prev_obs + self.step * self.prev_act
        if ev["new_episode"]:
            self.prev_act = None
        self.prev_obs = obs
        return ev

    def commit(self, action):
        self.prev_act = np.asarray(action, dtype=float)
        return np.asarray(action, dtype=np.float32)


# --------------------------------------------------------------------------- #
# Final policy: learned free-space prior + contact map + value-field planning
# --------------------------------------------------------------------------- #
_NEIGHBORS = [(-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
              (-1, -1, 1.4142), (-1, 1, 1.4142), (1, -1, 1.4142), (1, 1, 1.4142)]


class StudentPolicy:
    """Behavioural-cloning prior + contact mapping + replanning.

    * Free space: follow the BC policy learned from the (obs, action) demos
      (default: SymmetricLinearPolicy = linear BC + D4 augmentation, no bias).
    * Every detected collision marks the attempted end point (goal-centred frame)
      as an obstacle disc on an occupancy grid; unknown cells are assumed free
      (the free-space assumption of D* Lite).
    * When the straight line to the goal is not known to be free, compute a
      cost-to-go field V on the grid (Dijkstra from the goal, i.e. value iteration
      on the map) and pick, among `n_dirs` unit actions, the one whose *actual*
      0.08 end point has the lowest V. Scoring the true end point (not the next
      grid waypoint) is what makes the plan executable with a large step.

    No obstacle layout, target or scenario information is used: act() only sees
    the relative observation [dx, dy].
    """

    def __init__(self, X, Y, prior=None, step=0.08, res=0.02, extent=1.04,
                 inflate=0.06, n_dirs=48, jump=2.0, stall=0.5):
        self.prior = prior if prior is not None else SymmetricLinearPolicy(X, Y)
        self.tracker = ContactTracker(step=step, jump=jump, stall=stall)
        self.step, self.res, self.extent, self.inflate = step, res, extent, inflate
        self.N = int(round(2 * extent / res)) + 1
        r = int(np.ceil(inflate / res))
        ii, jj = np.mgrid[-r:r + 1, -r:r + 1]
        keep = (ii ** 2 + jj ** 2) * res ** 2 <= inflate ** 2 + 1e-12
        self._disc = np.stack([ii[keep], jj[keep]], axis=1)
        ang = np.linspace(0, 2 * np.pi, n_dirs, endpoint=False)
        self._dirs = np.stack([np.cos(ang), np.sin(ang)], axis=1)
        self.stats = {"replans": 0, "hits": 0}
        self._reset()

    # ---- map helpers ------------------------------------------------------ #
    def _reset(self):
        self.blocked = np.zeros((self.N, self.N), dtype=bool)
        self.visited = np.zeros((self.N, self.N), dtype=bool)
        self.V = None
        self.hits = []

    def _cells(self, pts):
        c = np.round((np.atleast_2d(pts) + self.extent) / self.res).astype(int)
        ok = np.all((c >= 0) & (c < self.N), axis=1)
        return c, ok

    def _mark_hit(self, q):
        c, _ = self._cells(q)
        cells = c[0] + self._disc
        ok = np.all((cells >= 0) & (cells < self.N), axis=1)
        cells = cells[ok]
        self.blocked[cells[:, 0], cells[:, 1]] = True
        self.blocked &= ~self.visited  # places we have stood on are free
        self.hits.append(np.asarray(q, float))
        self.V = None
        self.stats["hits"] += 1

    def _segment_free(self, a, b):
        n = max(2, int(np.ceil(np.linalg.norm(b - a) / (0.5 * self.res))) + 1)
        pts = a + np.linspace(0, 1, n)[:, None] * (b - a)
        c, ok = self._cells(pts)
        if not ok.all():
            return False
        return not self.blocked[c[:, 0], c[:, 1]].any()

    def _value_field(self):
        """Dijkstra from the goal cell over unblocked cells (8-connected)."""
        N, blocked = self.N, self.blocked
        V = np.full((N, N), np.inf)
        g = int(round(self.extent / self.res))
        V[g, g] = 0.0
        heap = [(0.0, g, g)]
        while heap:
            d, i, j = heapq.heappop(heap)
            if d > V[i, j]:
                continue
            for di, dj, w in _NEIGHBORS:
                ni, nj = i + di, j + dj
                if 0 <= ni < N and 0 <= nj < N and not blocked[ni, nj]:
                    nd = d + w
                    if nd < V[ni, nj]:
                        V[ni, nj] = nd
                        heapq.heappush(heap, (nd, ni, nj))
        self.stats["replans"] += 1
        return V * self.res

    # ---- policy ------------------------------------------------------------ #
    def act(self, obs):
        obs = np.asarray(obs, dtype=float)
        ev = self.tracker.observe(obs)
        if ev["new_episode"]:
            self._reset()
        p = -obs  # robot position in the goal-centred frame
        c, ok = self._cells(p)
        if ok[0]:
            self.visited[c[0, 0], c[0, 1]] = True
            self.blocked[c[0, 0], c[0, 1]] = False
        if ev["collided"]:
            self._mark_hit(ev["hit_point"])

        if not self.hits or self._segment_free(p, np.zeros(2)):
            return self.tracker.commit(self.prior.act(obs.astype(np.float32)))

        if self.V is None:
            self.V = self._value_field()
        ends = p + self.step * self._dirs
        mids = p + 0.5 * self.step * self._dirs
        ce, oke = self._cells(ends)
        cm, okm = self._cells(mids)
        score = np.full(len(ends), np.inf)
        good = oke & okm
        idx = np.where(good)[0]
        if len(idx):
            free = ~self.blocked[ce[idx, 0], ce[idx, 1]] & ~self.blocked[cm[idx, 0], cm[idx, 1]]
            idx = idx[free]
            score[idx] = self.V[ce[idx, 0], ce[idx, 1]] + 1e-3 * np.linalg.norm(ends[idx], axis=1)
        k = int(np.argmin(score))
        if not np.isfinite(score[k]):  # boxed in by the (inflated) map: fall back
            return self.tracker.commit(self.prior.act(obs.astype(np.float32)))
        return self.tracker.commit(_unit(self._dirs[k]))


# Implement StudentPolicy in this file (optional class name if you document it in your report).
# Training data: make_dataset() -> (X, Y), fixed demo budget unless you justify otherwise.
# At runtime, act(obs) receives obs = [target_x - robot_x, target_y - robot_y].

**单元格 3：写入对比用的策略 src/variants.py（MLP / kNN / 3a / 3b）**

In [ ]:
%%writefile src/variants.py
"""Alternative policies used for comparison / ablation (not the final submission).

- MLPPolicy, KNNPolicy, LinearNoBiasPolicy: memoryless behavioural cloning variants.
- RandomEscapePolicy (3a): BC + random escape after a detected collision.
- BugPolicy (3b): BC + map-free wall following (Bug-style) after a collision.
"""
import numpy as np

from .policy import LinearPolicy, ContactTracker, _unit, d4_augment


class MLPD4Policy:
    """MLP BC trained on D4-augmented demos (8x data)."""

    def __init__(self, X, Y, **kw):
        Xa, Ya = d4_augment(X, Y)
        self.net = MLPPolicy(Xa, Ya, **kw)

    def act(self, obs):
        return self.net.act(obs)


# --------------------------------------------------------------------------- #
# Memoryless BC variants
# --------------------------------------------------------------------------- #
def _features(X):
    X = np.atleast_2d(np.asarray(X, dtype=float))
    d = np.linalg.norm(X, axis=1, keepdims=True)
    u = X / np.maximum(d, 1e-8)
    return np.concatenate([X, d, u], axis=1)


class LinearNoBiasPolicy:
    """Least squares without the bias column: a = W^T obs."""

    def __init__(self, X, Y):
        self.W = np.linalg.pinv(X) @ Y

    def act(self, obs):
        return _unit(np.asarray(obs, float) @ self.W)


class MLPPolicy:
    """2-hidden-layer tanh MLP trained with Adam on MSE (pure NumPy)."""

    def __init__(self, X, Y, hidden=64, epochs=3000, lr=3e-3, wd=1e-4, seed=0):
        rng = np.random.default_rng(seed)
        F = _features(X)
        self.mu, self.sd = F.mean(0), F.std(0) + 1e-8
        F = (F - self.mu) / self.sd
        sizes = [F.shape[1], hidden, hidden, Y.shape[1]]
        self.params = []
        for a, b in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(0, np.sqrt(1.0 / a), (a, b)), np.zeros(b)]
        m = [np.zeros_like(p) for p in self.params]
        v = [np.zeros_like(p) for p in self.params]
        b1, b2 = 0.9, 0.999
        for t in range(1, epochs + 1):
            out, cache = self._forward(F)
            g_out = 2 * (out - Y) / len(F)
            grads = self._backward(g_out, cache)
            for i, (p, g) in enumerate(zip(self.params, grads)):
                g = g + wd * p
                m[i] = b1 * m[i] + (1 - b1) * g
                v[i] = b2 * v[i] + (1 - b2) * g * g
                p -= lr * (m[i] / (1 - b1 ** t)) / (np.sqrt(v[i] / (1 - b2 ** t)) + 1e-8)
        self.train_mse = float(np.mean((self._forward(F)[0] - Y) ** 2))

    def _forward(self, F):
        W1, b1, W2, b2, W3, b3 = self.params
        h1 = np.tanh(F @ W1 + b1)
        h2 = np.tanh(h1 @ W2 + b2)
        return h2 @ W3 + b3, (F, h1, h2)

    def _backward(self, g, cache):
        F, h1, h2 = cache
        W1, b1, W2, b2, W3, b3 = self.params
        gW3, gb3 = h2.T @ g, g.sum(0)
        g2 = (g @ W3.T) * (1 - h2 ** 2)
        gW2, gb2 = h1.T @ g2, g2.sum(0)
        g1 = (g2 @ W2.T) * (1 - h1 ** 2)
        gW1, gb1 = F.T @ g1, g1.sum(0)
        return [gW1, gb1, gW2, gb2, gW3, gb3]

    def act(self, obs):
        F = (_features(obs) - self.mu) / self.sd
        return _unit(self._forward(F)[0][0])


class KNNPolicy:
    """Average the actions of the k nearest demonstrations (in obs space)."""

    def __init__(self, X, Y, k=15):
        self.X, self.Y, self.k = np.asarray(X, float), np.asarray(Y, float), k

    def act(self, obs):
        d = np.sum((self.X - np.asarray(obs, float)) ** 2, axis=1)
        idx = np.argpartition(d, self.k)[: self.k]
        return _unit(self.Y[idx].mean(0))


# --------------------------------------------------------------------------- #
# 3a: random escape
# --------------------------------------------------------------------------- #
class RandomEscapePolicy:
    """BC prior; after a collision, move in a random direction for a few steps."""

    def __init__(self, X, Y, escape_steps=3, seed=0, prior=None):
        self.prior = prior if prior is not None else LinearPolicy(X, Y)
        self.tracker = ContactTracker()
        self.escape_steps = escape_steps
        self.rng = np.random.default_rng(seed)
        self.left, self.dir = 0, None

    def act(self, obs):
        ev = self.tracker.observe(obs)
        if ev["new_episode"]:
            self.left = 0
        if ev["collided"]:
            base = np.arctan2(obs[1], obs[0])
            ang = base + self.rng.choice([-1, 1]) * self.rng.uniform(np.pi / 3, 5 * np.pi / 6)
            self.dir = np.array([np.cos(ang), np.sin(ang)])
            self.left = self.escape_steps
        if self.left > 0:
            self.left -= 1
            a = self.dir
        else:
            a = self.prior.act(obs)
        return self.tracker.commit(_unit(a))


# --------------------------------------------------------------------------- #
# 3b: Bug-style wall following (no map)
# --------------------------------------------------------------------------- #
def _rot(v, ang):
    c, s = np.cos(ang), np.sin(ang)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])


class BugPolicy:
    """Map-free wall following, in the spirit of Bug2 (Lumelsky & Stepanov).

    GO mode: follow the BC prior (≈ head to goal).
    On a collision: enter FOLLOW mode, remember the distance at the hit point and
    turn away from the obstacle (always to the same side).
    FOLLOW mode: after a successful step, turn back toward the wall by
    `probe` radians to keep contact; after a blocked step turn away by `turn`.
    Leave FOLLOW (back to GO) once we are `leave_margin` closer to the goal
    than at the hit point.
    """

    def __init__(self, X, Y, side=1, turn=np.pi / 4, probe=np.pi / 4,
                 leave_margin=0.04, prior=None):
        self.prior = prior if prior is not None else LinearPolicy(X, Y)
        self.tracker = ContactTracker()
        self.side, self.turn, self.probe = side, turn, probe
        self.leave_margin = leave_margin
        self._reset()

    def _reset(self):
        self.mode, self.heading, self.d_hit = "GO", None, None

    def act(self, obs):
        obs = np.asarray(obs, float)
        ev = self.tracker.observe(obs)
        if ev["new_episode"]:
            self._reset()
        dist = np.linalg.norm(obs)
        if self.mode == "GO":
            if ev["collided"]:
                self.mode, self.d_hit = "FOLLOW", dist
                self.heading = _rot(self.tracker.prev_act, self.side * np.pi / 2)
            else:
                return self.tracker.commit(self.prior.act(obs))
        else:  # FOLLOW
            if ev["collided"]:
                self.heading = _rot(self.heading, self.side * self.turn)
            else:
                if dist < self.d_hit - self.leave_margin:
                    self.mode = "GO"
                    return self.tracker.commit(self.prior.act(obs))
                self.heading = _rot(self.heading, -self.side * self.probe)
        return self.tracker.commit(_unit(self.heading))

In [ ]:
# 单元格 4：生成示范数据，对比 baseline 和 StudentPolicy（public 集）
import importlib, time
import numpy as np
import src.policy, src.variants
importlib.reload(src.policy); importlib.reload(src.variants)   
from src.env import make_dataset
from src.evaluate import evaluate, make_scenarios
from src.policy import LinearPolicy, SymmetricLinearPolicy, StudentPolicy
from src.variants import LinearNoBiasPolicy, MLPPolicy, KNNPolicy, RandomEscapePolicy, BugPolicy

t = time.time()
X, Y = make_dataset(n=4000, seed=42)          
public = make_scenarios(300, seed=1)          
print(f'demos: {X.shape}, {time.time() - t:.0f}s')


def blocked_steps(res):
    # 每回合被挡住的步数（位置没变的步数）
    return np.mean([np.sum(np.all(np.diff(tr['positions'], axis=0) == 0, axis=1)) for tr in res['trajectories']])


def report(name, res):
    print(f"{name:26s} success {res['success_rate']:.3f} | dist {res['mean_final_distance']:.3f} | "
          f"steps {res['mean_steps']:5.1f} | collision-ep {res['collision_episode_rate']:.3f} | "
          f"blocked/ep {blocked_steps(res):5.2f}")


res_base = evaluate(LinearPolicy(X, Y), public)
res_stu = evaluate(StudentPolicy(X, Y), public)
report('baseline (linear BC)', res_base)
report('StudentPolicy (3c)', res_stu)

In [ ]:
# 单元格 5：所有策略的对比表（public 集）
policies = {
    'linear BC (baseline)': lambda: LinearPolicy(X, Y),
    'linear BC, no bias': lambda: LinearNoBiasPolicy(X, Y),
    'linear BC + D4': lambda: SymmetricLinearPolicy(X, Y),
    'MLP BC': lambda: MLPPolicy(X, Y),         
    'kNN BC': lambda: KNNPolicy(X, Y),
    '3a random escape': lambda: RandomEscapePolicy(X, Y),
    '3b bug / wall-follow': lambda: BugPolicy(X, Y),
    '3c StudentPolicy': lambda: StudentPolicy(X, Y),
}
results = {}
for name, make_pol in policies.items():
    results[name] = evaluate(make_pol(), public)
    report(name, results[name])

In [ ]:
# 单元格 6：三个诊断（expert 卡死 / 栅格副产品 / 小数据时的 bias）
from src.env import PointRobotEnv
from src.obstacles import expert_direction

# (1) 直接执行拿着真实地图的 A* expert
env = PointRobotEnv(seed=123)
fails = stuck = 0
for sc in public:
    env.reset(start=sc['start'], target=sc['target'], blocks=sc['blocks'])
    P = [env.pos.copy()]
    for _ in range(80):
        _, _, done, info = env.step(expert_direction(env.pos, env.target, env.blocks))
        P.append(env.pos.copy())
        if done:
            break
    if not info['success']:
        fails += 1
        stuck += np.allclose(P[-1], P[-10])
print(f'[1] A* expert executed: success {1 - fails / 300:.3f}; {stuck}/{fails} failures end stuck in place')

# (2) 没有障碍、没有噪声时，expert 的方向和目标方向差多少
rng = np.random.default_rng(0)
A, G = [], []
for _ in range(2000):
    s, tg = rng.uniform(0.05, 0.95, 2), rng.uniform(0.05, 0.95, 2)
    A.append(expert_direction(s, tg, np.zeros((0, 4)))); G.append((tg - s) / np.linalg.norm(tg - s))
A, G = np.array(A), np.array(G)
dev = np.degrees(np.arccos(np.clip(np.sum(A * G, 1), -1, 1)))
dev_demo = np.degrees(np.arccos(np.clip(np.sum(Y * X / np.linalg.norm(X, axis=1, keepdims=True), 1), -1, 1)))
ea = np.degrees(np.arctan2(A[:, 1], A[:, 0])) % 90
ga = np.degrees(np.arctan2(G[:, 1], G[:, 0])) % 90
print(f'[2] > 30 deg off goal direction: demos {np.mean(dev_demo > 30) * 100:.1f}%, '
      f'expert with NO obstacles {np.mean(dev > 30) * 100:.1f}%')
print(f'    first step within 8 deg of a diagonal: expert {np.mean(np.abs(ea - 45) < 8) * 100:.0f}%, '
      f'goal directions {np.mean(np.abs(ga - 45) < 8) * 100:.0f}%')

# (3) 示范很少时，普通线性 BC 的 bias；D4 对称版本的 W
for n in [10, 30, 300, 4000]:
    print(f'[3] n={n:5d}: linear BC bias = {np.round(LinearPolicy(X[:n], Y[:n]).W[2], 3)}, '
          f'D4 W = {np.round(SymmetricLinearPolicy(X[:n], Y[:n]).W, 3).tolist()}')

In [ ]:
# 单元格 7：四种策略走同样的场景（红叉 = StudentPolicy 发现的接触点）
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

COLORS = {'baseline': '#eb6834', '3a': '#eda100', '3b': '#1baf7a', '3c': '#2a78d6'}
MAKERS = {'baseline': lambda: LinearPolicy(X, Y), '3a': lambda: RandomEscapePolicy(X, Y),
          '3b': lambda: BugPolicy(X, Y), '3c': lambda: StudentPolicy(X, Y)}


def draw(ax, sc, pos, color, title, hits=(), r=0.06):
    for x0, y0, x1, y1 in sc['blocks']:
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fc='0.85', ec='0.6'))
    for q in hits:
        ax.add_patch(Circle(sc['target'] + q, r, fc='#e34948', alpha=0.2, ec='none'))
        ax.plot(*(sc['target'] + q), 'x', color='#d03b3b')
    ax.plot(pos[:, 0], pos[:, 1], '-o', ms=3, color=color)
    ax.plot(*sc['start'], 'ko')
    ax.plot(*sc['target'], '*', ms=14, color='#2a78d6')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)


fails = [i for i, tr in enumerate(res_base['trajectories'])
         if np.linalg.norm(tr['positions'][-1] - tr['target']) >= 0.05]
print('baseline 失败的场景编号：', fails)
SHOW = fails[:2]          # 可以改成任意场景编号

fig, axs = plt.subplots(len(SHOW), 4, figsize=(15, 3.9 * len(SHOW)), squeeze=False)
for r, i in enumerate(SHOW):
    sc = public[i]
    for c, name in enumerate(['baseline', '3a', '3b', '3c']):
        pol = MAKERS[name]()
        res = evaluate(pol, [sc])
        pos = res['trajectories'][0]['positions']
        blocked = int(np.sum(np.all(np.diff(pos, axis=0) == 0, axis=1)))
        ok = 'success' if res['success_rate'] > 0 else 'FAIL'
        draw(axs[r, c], sc, pos, COLORS[name], f'{name} | scenario {i}: {ok}, {len(pos) - 1} steps, {blocked} blocked',
             hits=getattr(pol, 'hits', []), r=getattr(pol, 'inflate', 0.06))
plt.tight_layout(); plt.show()

In [ ]:
# 单元格 8：调参——接触点的膨胀半径 inflate
val_dense = make_scenarios(300, n_blocks=(5, 8), seed=100)
val_noise = make_scenarios(300, seed=100)
print('inflate | success  steps  blocked/ep | steps(noise)  blocked/ep(noise)')
for inflate in [0.02, 0.03, 0.04, 0.06, 0.08]:
    r1 = evaluate(StudentPolicy(X, Y, inflate=inflate), val_dense)
    r2 = evaluate(StudentPolicy(X, Y, inflate=inflate), val_noise, noise_std=0.02)
    print(f"  {inflate:.2f}  |  {r1['success_rate']:.3f}   {r1['mean_steps']:.2f}    {blocked_steps(r1):.2f}     |"
          f"     {r2['mean_steps']:.2f}          {blocked_steps(r2):.2f}")

In [ ]:
# 单元格 9：鲁棒性测试（更多障碍 / 观测噪声 / 动作噪声 / 更短步数上限）
SEEDS = [11]
sweeps = {
    'number of blocks': ([(2, 4), (4, 6), (6, 8), (8, 10)], lambda v: (dict(n_blocks=v), {})),
    'observation noise': ([0.0, 0.01, 0.02, 0.03], lambda v: ({}, dict(noise_std=v))),
    'action noise': ([0.0, 0.1, 0.2, 0.3], lambda v: ({}, dict(action_noise_std=v))),
    'max steps (4-6 blocks)': ([10, 15, 20, 40, 80], lambda v: (dict(n_blocks=(4, 6)), dict(max_steps=v))),
}
sweep_res = {}
t = time.time()
for sname, (levels, cfg) in sweeps.items():
    for lv in levels:
        skw, ekw = cfg(lv)
        for s in SEEDS:
            scen = make_scenarios(300, seed=s, **skw)
            for pname, mk in MAKERS.items():
                sweep_res.setdefault((sname, str(lv), pname), []).append(evaluate(mk(), scen, **ekw)['success_rate'])
    print(f'{sname}: done ({time.time() - t:.0f}s)')

fig, axs = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, (sname, (levels, _)) in zip(axs, sweeps.items()):
    for pname in MAKERS:
        m = [np.mean(sweep_res[(sname, str(lv), pname)]) for lv in levels]
        ax.plot(range(len(levels)), m, '-o', color=COLORS[pname], label=pname)
    ax.set_xticks(range(len(levels))); ax.set_xticklabels([str(lv) for lv in levels])
    ax.set_title(sname); ax.set_ylim(0.45, 1.02); ax.grid(alpha=0.3)
axs[0].set_ylabel('success rate'); axs[0].legend(); plt.tight_layout(); plt.show()

print('\nsuccess rate table')
for sname, (levels, _) in sweeps.items():
    for lv in levels:
        print(f'{sname:24s} {str(lv):8s} ' + '  '.join(
            f'{p} {np.mean(sweep_res[(sname, str(lv), p)]):.3f}' for p in MAKERS))

In [ ]:
# 单元格 10：压力测试（6–8 个障碍 + 观测噪声 0.01 + 动作噪声 0.1 + 40 步上限）
STRESS_SEEDS = [21]       # 报告里用的是 [21, 22, 23]
for pname, mk in MAKERS.items():
    rs = [evaluate(mk(), make_scenarios(300, n_blocks=(6, 8), seed=s),
                   noise_std=0.01, action_noise_std=0.1, max_steps=40) for s in STRESS_SEEDS]
    print(f"{pname:9s} success {np.mean([r['success_rate'] for r in rs]):.3f} | "
          f"steps {np.mean([r['mean_steps'] for r in rs]):.1f}")

In [ ]:
# 单元格 11：原规则 vs 严格碰撞检测（模拟器只检查落点）
import src.evaluate as ev
from src.env import PointRobotEnv
from src.obstacles import point_in_rects

U = np.linspace(0, 1, 42)[1:-1]          # 每步检查 40 个中间点（间距 0.002）

def seg_hits(a, b, blocks):
    return any(point_in_rects(a + u * (b - a), blocks) for u in U)


class StrictEnv(PointRobotEnv):
    def step(self, action):
        action = np.asarray(action, dtype=float)
        if self.action_noise_std > 0:
            action = action + self.rng.normal(0, self.action_noise_std, size=2)
        n = np.linalg.norm(action)
        action = action / n if n > 1e-8 else np.zeros(2)
        proposed = np.clip(self.pos + self.step_size * action, 0.0, 1.0)
        collision = point_in_rects(proposed, self.blocks) or seg_hits(self.pos, proposed, self.blocks)
        if collision:
            self.collisions += 1
        else:
            self.pos = proposed
        self.t += 1
        dist = float(np.linalg.norm(self.pos - self.target))
        in_obstacle = point_in_rects(self.pos, self.blocks)
        success = dist < self.success_dist and not in_obstacle
        done = success or self.t >= self.max_steps
        return self.observe(), (1.0 if success else -dist), done, {
            'success': success, 'distance': dist, 'collision': collision, 'in_obstacle': in_obstacle}


def crossing_successes(res):
    # 原规则下成功、但至少有一步斜切过障碍的回合数
    n = 0
    for tr in res['trajectories']:
        P, B = tr['positions'], tr['blocks']
        if np.linalg.norm(P[-1] - tr['target']) < 0.05 and any(
                seg_hits(P[k], P[k + 1], B) for k in range(len(P) - 1) if not np.allclose(P[k], P[k + 1])):
            n += 1
    return n


STRICT_SETS = {
    'public (seed 1)': (public, {}),
    'stress (seed 21)': (make_scenarios(300, n_blocks=(6, 8), seed=21),
                         dict(noise_std=0.01, action_noise_std=0.1, max_steps=40)),
}
try:
    for sname, (scen, kw) in STRICT_SETS.items():
        print(f'=== {sname}')
        for pname, mk in MAKERS.items():
            ev.PointRobotEnv = PointRobotEnv
            r0 = evaluate(mk(), scen, **kw)
            ev.PointRobotEnv = StrictEnv
            r1 = evaluate(mk(), scen, **kw)
            print(f"  {pname:9s} success {r0['success_rate']:.3f} -> {r1['success_rate']:.3f} (strict) | "
                  f"steps {r0['mean_steps']:4.1f} -> {r1['mean_steps']:4.1f} | "
                  f"successes that cut a block corner (original rule): {crossing_successes(r0)}/{round(r0['success_rate'] * len(scen))}")
finally:
    ev.PointRobotEnv = PointRobotEnv      

In [ ]:
# 单元格 12：示范数量实验（对称性让 10 条示范 = 4000 条）
for n in [10, 30, 100, 300, 1000, 4000]:
    s = [evaluate(P(X[:n], Y[:n]), public)['success_rate']
         for P in (LinearPolicy, SymmetricLinearPolicy,
                   lambda a, b: StudentPolicy(a, b, prior=LinearPolicy(a, b)), StudentPolicy)]
    print(f'n={n:5d} | linear BC {s[0]:.3f} | linear BC + D4 {s[1]:.3f} | '
          f'3c with plain linear prior {s[2]:.3f} | 3c (D4 prior) {s[3]:.3f}')

In [ ]:
# 单元格 13：动画演示——同一个场景，baseline 对比 StudentPolicy
from matplotlib import animation
from IPython.display import HTML

dense = make_scenarios(300, n_blocks=(6, 8), seed=12)
fails_dense = [i for i, sc in enumerate(dense) if evaluate(LinearPolicy(X, Y), [sc])['success_rate'] == 0]
sc = dense[fails_dense[5]]          # 换一个编号就换一个场景


def rollout(pol, sc):
    env = PointRobotEnv(seed=123)
    obs = env.reset(start=sc['start'], target=sc['target'], blocks=sc['blocks'])
    P, H = [env.pos.copy()], [[]]
    for _ in range(80):
        obs, _, done, _ = env.step(pol.act(obs))
        P.append(env.pos.copy())
        H.append([sc['target'] + q for q in getattr(pol, 'hits', [])])
        if done:
            break
    return np.array(P), H


pb, _ = rollout(LinearPolicy(X, Y), sc)
stu = StudentPolicy(X, Y)
ps, hs = rollout(stu, sc)

fig, axs = plt.subplots(1, 2, figsize=(9, 4.6))
lines = []
for ax, name in zip(axs, ['baseline', '3c']):
    for x0, y0, x1, y1 in sc['blocks']:
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fc='0.85', ec='0.6'))
    ax.plot(*sc['start'], 'ko'); ax.plot(*sc['target'], '*', ms=14, color='#2a78d6')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    lines.append((ax.plot([], [], '-o', ms=3, color=COLORS[name])[0], ax.set_title(name)))
hit_marks = axs[1].plot([], [], 'x', color='#d03b3b', ms=9, mew=2)[0]
discs = []


def frame(k):
    for (ln, title), P, name in zip(lines, [pb, ps], ['baseline', 'StudentPolicy (3c)']):
        j = min(k, len(P) - 1)
        ln.set_data(P[:j + 1, 0], P[:j + 1, 1])
        blocked = int(np.sum(np.all(np.diff(P[:j + 1], axis=0) == 0, axis=1)))
        title.set_text(f'{name}: step {j}, blocked {blocked}')
    h = hs[min(k, len(hs) - 1)]
    hit_marks.set_data([q[0] for q in h], [q[1] for q in h])
    while len(discs) < len(h):
        discs.append(axs[1].add_patch(Circle(h[len(discs)], stu.inflate, fc='#e34948', alpha=0.2, ec='none')))
    return []


ani = animation.FuncAnimation(fig, frame, frames=40, interval=300)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# 单元格 14：下载代码（src/ 和 notebook），用于提交到 GitHub或者其他地方。
!zip -qr /content/my_solution.zip src colab
try:
    from google.colab import files
    files.download('/content/my_solution.zip')
except ImportError:
    print('不在 Colab 里：文件已保存到 /content/my_solution.zip')